In [1]:
from src.data_utils import load_documents, chunker
from src.retrievers.hybrid import HybridRetriever
from src.retrievers.bm25 import Bm25Retriever
from src.retrievers.dense import DenseRetriever
from src.generate import generate


documents = load_documents(file_path="/home/nick/new-speedrun/data/2038.pdf")
chunks = chunker(documents=documents, chunk_size=301, chunk_overlap=30.1)

/home/nick/new-speedrun/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
hybrid = HybridRetriever(
    bm25=Bm25Retriever(),
    dense=DenseRetriever(),
    bm25_weight=0.1,
    dense_weight=0.7,
    candidates=40,
    rrf_k=60
)


hybrid.add_documents(documents=chunks)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9165.45it/s]


In [3]:
from src.generate import generate

def rag_answer(query: str, k: int) -> str:
    if k < 1:
        raise ValueError("k must be at least 1")
    retrieved = hybrid.retrieve(query, top_k=k)
    context_chunks = [doc.page_content for doc in retrieved]
    answer = generate(query, context_chunks)
    return answer


In [4]:
test_queries = [
    'Does Sterling Bronson have a mag?',
    'What did Darren Walsh do for Work in 2027?',
    'Which pub does Winston Reed regularly visit?',
    'What town is this novel set in?',
    'Who is Grace?',
    'Which chapter first introduces Cassie May?',
    'Which social or occupational class does Cassie belong to?',
    'What are cats in this novel?',
    'what are transistors?',
    'does sterling own the transistors?',
    'what are the people who own the transistors referred to?'
]

for query in test_queries:
    print(f"Question: {query}")
    print(f"Answer: {rag_answer(query, k=20)}")
    print()




Question: Does Sterling Bronson have a mag?
Answer: Yes, Sterling Bronson has a Mag. It is mentioned that he had built one himself that had learned the figures and every board from every Assay in every town.

Question: What did Darren Walsh do for Work in 2027?
Answer: Darren Walsh was a scaffolder in 2027.

Question: Which pub does Winston Reed regularly visit?
Answer: Winston Reed regularly visits "the Crown."

Question: What town is this novel set in?
Answer: The novel is set in the town of Brackton.

Question: Who is Grace?
Answer: Grace is a Mag, a type of companion or entity that lives with Reed. She has been learning and understanding him for years, providing companionship and support. Grace is characterized by her ability to respond in exactly the right way to Reed's actions and emotions, making her an integral part of his life.

Question: Which chapter first introduces Cassie May?
Answer: Cassie May is first introduced in Chapter Three.

Question: Which social or occupational 